# L13: 短期记忆设计

> 理解 Agent 如何「记住」对话历史

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from datetime import datetime
from typing import List, Optional, Dict, Any

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.llm.models import Message
from backend.memory.short_term import ShortTermMemory
print("✓ 模块导入成功")

## 13.1 什么是短期记忆？

### 定义

**短期记忆 (Short-term Memory)** = 当前对话的消息历史

### 为什么需要短期记忆？

```
用户: "我想买一台电脑"
Agent: "好的，您有什么具体需求吗？"

用户: "预算1万左右"  ← 没有记忆，Agent 不知道这是关于什么的
```

### 短期记忆的特点

| 特点 | 说明 |
|------|------|
| **临时存储** | 会话结束即清空 |
| **快速增长** | 每轮对话都会增加 |
| **窗口限制** | 不能超过 Context Window |
| **需要管理** | 必须裁剪、压缩、总结 |

## 13.2 查看项目中的短期记忆实现

In [ ]:
import inspect
from backend.memory.short_term import ShortTermMemory

print("=== ShortTermMemory 源码 ===\n")
print(inspect.getsource(ShortTermMemory))

## 13.3 短期记忆的基本操作

In [ ]:
async def demo_basic_operations():
    """演示短期记忆的基本操作"""
    
    # 创建短期记忆实例
    memory = ShortTermMemory(max_messages=100)
    
    print("=== 1. 添加消息 ===\n")
    
    # 添加消息
    memory.add_message(Message(role="user", content="你好"))
    memory.add_message(Message(role="assistant", content="你好！有什么可以帮助你的吗？"))
    memory.add_message(Message(role="user", content="我想了解 Python"))
    
    print(f"消息数量: {len(memory.get_messages())}")
    print(f"最近一条: {memory.get_last_message().content}")
    
    print("\n=== 2. 获取历史 ===\n")
    
    # 获取所有消息
    messages = memory.get_messages()
    for i, msg in enumerate(messages, 1):
        print(f"{i}. [{msg.role}] {msg.content}")
    
    print("\n=== 3. Token 统计 ===\n")
    
    # 获取 token 统计
    stats = memory.get_token_stats()
    print(f"估算 Token 数: {stats['total_tokens']}")
    print(f"消息数量: {stats['message_count']}")
    
    print("\n=== 4. 清空记忆 ===\n")
    
    memory.clear()
    print(f"清空后消息数量: {len(memory.get_messages())}")

await demo_basic_operations()

## 13.4 窗口管理策略

### 问题：消息太多怎么办？

当对话历史超过 Context Window 时，需要裁剪。

### 策略 1: 滑动窗口 (Sliding Window)

保留最近的 N 条消息，删除旧的。

In [ ]:
class SlidingWindowMemory:
    """滑动窗口记忆管理"""
    
    def __init__(self, window_size: int = 10):
        self.window_size = window_size
        self.messages: List[Message] = []
    
    def add_message(self, message: Message):
        """添加消息，超过窗口大小时删除最旧的"""
        self.messages.append(message)
        
        # 超过窗口大小时裁剪
        if len(self.messages) > self.window_size:
            # 总是保留 system 消息
            system_msgs = [m for m in self.messages if m.role == "system"]
            other_msgs = [m for m in self.messages if m.role != "system"]
            
            # 裁剪非 system 消息
            other_msgs = other_msgs[-(self.window_size - len(system_msgs)):]
            
            self.messages = system_msgs + other_msgs
    
    def get_messages(self) -> List[Message]:
        return self.messages.copy()
    
    def clear(self):
        self.messages = []

# 测试滑动窗口
async def test_sliding_window():
    memory = SlidingWindowMemory(window_size=5)
    
    print("=== 滑动窗口测试 (窗口大小=5) ===\n")
    
    # 添加 system 消息
    memory.add_message(Message(role="system", content="你是一个助手"))
    
    # 添加 10 条对话
    for i in range(1, 11):
        memory.add_message(Message(role="user", content=f"消息 {i}"))
        if i % 2 == 0:
            memory.add_message(Message(role="assistant", content=f"回复 {i}"))
        
        print(f"添加消息 {i} 后，共 {len(memory.get_messages())} 条消息")
        
        if i == 5 or i == 10:
            print("  当前消息:")
            for msg in memory.get_messages():
                print(f"    - [{msg.role}] {msg.content}")
            print()

await test_sliding_window()

### 策略 2: 智能裁剪 (Intelligent Pruning)

根据重要性保留消息，而不仅仅是按时间。

In [ ]:
class IntelligentPruningMemory:
    """智能裁剪记忆管理"""
    
    def __init__(self, max_tokens: int = 1000):
        self.max_tokens = max_tokens
        self.messages: List[Message] = []
    
    def _estimate_tokens(self, text: str) -> int:
        """粗略估算 token 数量"""
        return len(text) // 3  # 粗略估算
    
    def _get_message_score(self, message: Message) -> float:
        """
        计算消息的重要性分数
        """
        score = 0.0
        
        # system 消息最重要
        if message.role == "system":
            score += 1000
        
        # 最近的消息更重要
        index = self.messages.index(message) if message in self.messages else 0
        score += (len(self.messages) - index) * 10
        
        # 包含关键词的消息更重要
        keywords = ["重要", "记住", "关键", "必须", "目标", "要求"]
        for kw in keywords:
            if kw in message.content:
                score += 50
        
        # 用户问题比助手回复重要
        if message.role == "user":
            score += 20
        
        return score
    
    def add_message(self, message: Message):
        """添加消息，根据重要性裁剪"""
        self.messages.append(message)
        
        # 检查是否超过 token 限制
        while self._get_total_tokens() > self.max_tokens:
            # 找到重要性最低的消息（跳过 system）
            candidates = [
                (i, self._get_message_score(m)) 
                for i, m in enumerate(self.messages) 
                if m.role != "system"
            ]
            
            if not candidates:
                break
            
            # 删除分数最低的
            candidates.sort(key=lambda x: x[1])
            del self.messages[candidates[0][0]]
    
    def _get_total_tokens(self) -> int:
        """计算总 token 数"""
        return sum(self._estimate_tokens(m.content) for m in self.messages)
    
    def get_messages(self) -> List[Message]:
        return self.messages.copy()
    
    def get_token_stats(self) -> Dict[str, int]:
        return {
            "total_tokens": self._get_total_tokens(),
            "message_count": len(self.messages),
            "max_tokens": self.max_tokens
        }

# 测试智能裁剪
async def test_intelligent_pruning():
    memory = IntelligentPruningMemory(max_tokens=200)  # 小限制便于演示
    
    print("=== 智能裁剪测试 (最大 200 tokens) ===\n")
    
    # 添加 system 消息
    memory.add_message(Message(role="system", content="你是一个重要的助手"))
    
    # 添加各种消息
    messages = [
        ("user", "你好"),
        ("assistant", "你好！"),
        ("user", "请记住这个重要的信息"),  # 包含关键词
        ("assistant", "好的，我记住了"),
        ("user", "今天天气怎么样"),
        ("assistant", "今天天气不错"),
        ("user", "另一个普通问题"),
        ("assistant", "这是回答"),
    ]
    
    for role, content in messages:
        memory.add_message(Message(role=role, content=content))
        stats = memory.get_token_stats()
        print(f"添加: [{role}] {content}")
        print(f"  Token: {stats['total_tokens']}/{stats['max_tokens']}, 消息数: {stats['message_count']}")
    
    print("\n最终保留的消息:")
    for msg in memory.get_messages():
        score = memory._get_message_score(msg)
        print(f"  - [{msg.role}] {msg} (分数: {score:.1f})")

await test_intelligent_pruning()

### 策略 3: 摘要压缩 (Summary Compression)

将旧消息压缩成摘要。

In [ ]:
class SummaryCompressionMemory:
    """摘要压缩记忆管理"""
    
    def __init__(self, max_messages: int = 5, summary_threshold: int = 8):
        self.max_messages = max_messages
        self.summary_threshold = summary_threshold
        self.messages: List[Message] = []
        self.summary: str = ""
    
    def add_message(self, message: Message):
        """添加消息，超过阈值时压缩旧消息"""
        self.messages.append(message)
        
        # 检查是否需要压缩
        if len(self.messages) > self.summary_threshold:
            self._compress_old_messages()
    
    def _compress_old_messages(self):
        """
        压缩旧消息为摘要
        实际应用中应使用 LLM 生成摘要
        """
        # 保留最近的几条消息
        recent = self.messages[-(self.max_messages // 2):]
        
        # 旧消息生成摘要（简化版）
        old_messages = self.messages[:-(self.max_messages // 2)]
        if old_messages:
            old_summary = "; ".join([
                f"{m.role}: {m.content[:30]}..." 
                for m in old_messages
            ])
            
            if self.summary:
                self.summary += f"; {old_summary}"
            else:
                self.summary = f"历史对话: {old_summary}"
        
        # 添加摘要消息到开头
        self.messages = [Message(role="system", content=f"[摘要] {self.summary}")] + recent
    
    def get_messages(self) -> List[Message]:
        return self.messages.copy()
    
    def clear_summary(self):
        """清除摘要"""
        self.summary = ""
        # 移除系统摘要消息
        self.messages = [m for m in self.messages if not m.content.startswith("[摘要]")]

# 测试摘要压缩
async def test_summary_compression():
    memory = SummaryCompressionMemory(max_messages=5, summary_threshold=5)
    
    print("=== 摘要压缩测试 (最大5条, 阈值5条) ===\n")
    
    # 添加消息
    for i in range(1, 8):
        memory.add_message(Message(role="user", content=f"消息 {i}"))
        print(f"添加消息 {i} 后，共 {len(memory.get_messages())} 条")
        
        if i == 5:
            print("  触发压缩！")
            for msg in memory.get_messages():
                print(f"    - {msg.content[:60]}..." if len(msg.content) > 60 else f"    - {msg.content}")
    
    print("\n最终消息:")
    for msg in memory.get_messages():
        content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
        print(f"  [{msg.role}] {content}")

await test_summary_compression()

## 13.5 对话会话管理

### 多会话支持

In [ ]:
from typing import Dict

class SessionManager:
    """
    管理多个对话会话
    """
    
    def __init__(self):
        self.sessions: Dict[str, ShortTermMemory] = {}
    
    def get_or_create_session(self, session_id: str) -> ShortTermMemory:
        """获取或创建会话"""
        if session_id not in self.sessions:
            self.sessions[session_id] = ShortTermMemory()
        return self.sessions[session_id]
    
    def delete_session(self, session_id: str):
        """删除会话"""
        if session_id in self.sessions:
            del self.sessions[session_id]
    
    def list_sessions(self) -> List[str]:
        """列出所有会话 ID"""
        return list(self.sessions.keys())
    
    def get_session_stats(self, session_id: str) -> Optional[Dict]:
        """获取会话统计"""
        if session_id not in self.sessions:
            return None
        return self.sessions[session_id].get_token_stats()

# 测试会话管理
async def test_session_manager():
    manager = SessionManager()
    
    print("=== 会话管理测试 ===\n")
    
    # 创建多个会话
    sessions = ["user_123", "user_456", "user_789"]
    
    for session_id in sessions:
        session = manager.get_or_create_session(session_id)
        session.add_message(Message(
            role="user",
            content=f"Hello from {session_id}"
        ))
        session.add_message(Message(
            role="assistant",
            content=f"Nice to meet you, {session_id}!"
        ))
    
    print("会话列表:")
    for session_id in manager.list_sessions():
        stats = manager.get_session_stats(session_id)
        print(f"  - {session_id}: {stats['message_count']} 条消息")
    
    print("\n删除 user_456 会话...")
    manager.delete_session("user_456")
    print(f"剩余会话: {manager.list_sessions()}")

await test_session_manager()

## 13.6 常见面试问题

**Q: 短期记忆和长期记忆有什么区别？**
- A:
  | 特性 | 短期记忆 | 长期记忆 |
  |------|----------|----------|
  | 存储内容 | 当前对话历史 | 向量化的语义信息 |
  | 存储介质 | 内存列表 | 向量数据库 |
  | 持久化 | 会话结束即清空 | 持久存储 |
  | 检索方式 | 直接获取全部 | 语义相似度检索 |
  | 用途 | 保持对话上下文 | 跨会话知识复用 |

**Q: 如何估算 Token 数量？**
- A:
  1. 使用 tokenizer 库精确计算
  2. 粗略估算：英文 ≈ 4 字符/token，中文 ≈ 2-3 字符/token
  3. API 返回的 usage 中有精确数值

**Q: 滑动窗口的缺点是什么？**
- A:
  1. 可能丢失重要信息
  2. 长对话中上下文不连贯
  3. 无法追溯历史对话
  4. 解决方案：智能裁剪、摘要压缩

**Q: System Prompt 会因为裁剪丢失吗？**
- A:
  - 不应该！System Prompt 定义了 Agent 的行为
  - 裁剪时应该始终保留 system 角色的消息
  - 或者将 System Prompt 单独存储，每次请求都添加

**Q: 如何处理超长对话？**
- A:
  1. **滑动窗口**：保留最近 N 条
  2. **智能裁剪**：根据重要性选择保留
  3. **摘要压缩**：旧对话压缩成摘要
  4. **RAG 检索**：从向量库检索相关历史
  5. **混合策略**：最近消息 + 相关历史 + 摘要

---

## 总结

本课程学习了短期记忆的核心知识：

| 知识点 | 说明 |
|--------|------|
| **短期记忆** | 当前对话的消息历史 |
| **基本操作** | 添加、获取、清空、统计 |
| **滑动窗口** | 保留最近 N 条消息 |
| **智能裁剪** | 根据重要性选择保留 |
| **摘要压缩** | 旧消息压缩成摘要 |
| **会话管理** | 支持多用户多会话 |

**下一步**: L14 将学习向量数据库基础！